# Features: transferable inputs on DIII-D and MAST

Review notebook of the feature phase (`docs/FEATURES.md`). It reads the feature tables of
`scripts/build_features.py`, their build record, the scaler and the reports of
`scripts/profile_features.py` (checks X1 to X15) and `scripts/feature_linear_check.py` (X16),
and draws what the reports summarise. Nothing here fits a statistic: every comparison takes the
DIII-D training frames (the `train` list of the split file) as the reference, and the test
splits are compared against it.

Run order: the frame table with Part 0, the feature build and the profile come first; sections
1 to 8 are then reviewed. Section 9 reads the linear check and shows a note until it has run.
Section 10 gathers the numbers behind the decisions this phase feeds. Readings are carried to
`docs/findings.md` at step 8 of the procedure.

Cells are run manually, top to bottom. Section 2 reads the geometry columns of the first shot
of each public test folder; no other cell reads a raw shot file. Histograms are drawn from a
fixed-seed sample of training frames (`PLOT_FRAMES`) and every frame of the other splits up to
the same count; values outside the pooled 0.5th to 99.5th percentiles are drawn at the edges.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from fusion_eq import coil_pattern as cp, dataset, feature_checks as fc, features as fe, frame_table as ft, split as sp
from fusion_eq.coil_flux import GreenTable, element_outline
from fusion_eq.feature_scaler import RobustScaler
from fusion_eq.machines import machine_config

In [ ]:
DATA_ROOT = Path("../../data/raw/data")                  # geometry columns of one public test shot per machine (section 2)
TABLE_DIR = Path("../../data/processed/frame_table")     # frame table with Part 0
FEATURE_ROOT = Path("../../data/processed/features")     # unscaled feature tables and build.json
GREEN_DIR = Path("../../data/processed/green")           # grid tables
REPORT_ROOT = Path("../reports/features")                # scaler, layout clearances, profile and linear check
SPLIT_PATH = Path("../splits/split_v1.json")
CONFIG_PATH = Path("../configs/features_v1.yaml")

PLOT_FRAMES = 200_000       # frames per split drawn for histograms and scatter plots
SCATTER_FRAMES = 20_000     # frames per split in scatter plots
SEED = 0

In [ ]:
CONFIG = fe.FeatureConfig.load(CONFIG_PATH)
REPORT_DIR = REPORT_ROOT / f"v{CONFIG.version}"
FEATURE_DIR = fe.features_dir(CONFIG.version, FEATURE_ROOT)
BUILD = json.loads((FEATURE_DIR / "build.json").read_text())
PROFILE = json.loads((REPORT_DIR / "profile.json").read_text())
SCALER = RobustScaler.load(fe.scaler_path(CONFIG.version, REPORT_ROOT))
SPLIT = sp.read_split(SPLIT_PATH)
GROUPS = ("train", "validation", "diii_d_public_test", "mast_public_test")
MACHINE_OF = {"train": "DIII-D", "validation": "DIII-D", "diii_d_public_test": "DIII-D", "mast_public_test": "MAST"}
LABEL = {"train": "DIII-D train", "validation": "DIII-D validation", "diii_d_public_test": "DIII-D test",
         "mast_public_test": "MAST test"}
COLOR = {"train": "tab:blue", "validation": "tab:cyan", "diii_d_public_test": "tab:green", "mast_public_test": "tab:red"}
COMPARED = ("train", "diii_d_public_test", "mast_public_test")
POINTS = CONFIG.columns_of("coil_pt")
TERMS = [f"coil_{t}" for t in cp.TERM_NAMES]


def load(split):
    features = fe.read_features(split, CONFIG.version, CONFIG.all_columns(), None, FEATURE_ROOT)
    frames = ft.read_frame_table(split, ["shot", "frame", *fc.profile_frame_columns(dataset.SPLIT_MACHINE[split])],
                                 root=TABLE_DIR)
    return fc.join_frame_columns(features, frames)


d3d = load("diii_d_train")
DATA = {"train": d3d[d3d["shot"].isin(set(SPLIT["train"]))].reset_index(drop=True),
        "validation": d3d[d3d["shot"].isin(set(SPLIT["validation"]))].reset_index(drop=True),
        "diii_d_public_test": load("diii_d_public_test"), "mast_public_test": load("mast_public_test")}
del d3d
INDEX = pd.read_parquet(TABLE_DIR / "shot_index.parquet")
INDEX = INDEX[INDEX["split"] == "diii_d_train"].assign(shot=lambda t: t["shot"].astype(str)).set_index("shot")
DATA["train"]["ip_direction"] = DATA["train"]["shot"].map(INDEX["ip_direction"].astype("float64"))
PLOT = {g: t.iloc[fc.sample_rows(len(t), PLOT_FRAMES, SEED)] for g, t in DATA.items()}
for g, t in DATA.items():
    print(f"{LABEL[g]}: {len(t)} frames, {t['shot'].nunique()} shots; {len(PLOT[g])} drawn")


def overlaid(ax, column, groups=COMPARED, frames=None, bins=60):
    """Density histograms of one column per group of frames on shared bins."""
    frames = PLOT if frames is None else frames
    values = {g: frames[g][column].to_numpy(dtype=float) for g in groups}
    finite = np.concatenate([v[np.isfinite(v)] for v in values.values()])
    if finite.size == 0:
        ax.set_title(f"{column}: no finite value", fontsize=9)
        return
    lo, hi = np.percentile(finite, [0.5, 99.5])
    lo, hi = (lo - 0.5, hi + 0.5) if lo == hi else (lo, hi)
    edges = np.linspace(lo, hi, bins + 1)
    for g, v in values.items():
        v = v[np.isfinite(v)]
        if v.size:
            ax.hist(np.clip(v, lo, hi), bins=edges, density=True, histtype="step", color=COLOR.get(g),
                    label=f"{LABEL.get(g, g)} ({v.size})")
    ax.set_title(column, fontsize=9)
    ax.tick_params(labelsize=7)

## 1. Build summary and check table

The build record of the feature tables (configuration hash, constants and their verified
status, counts per split) and the check table of the profile. X13 is drawn in section 3 and X16
is in section 9.

In [ ]:
print(f"configuration hash at build {BUILD['config_hash'][:12]}, now {CONFIG.hash()[:12]}")
for machine, record in BUILD["resources"].items():
    constants = {k: f"{v['value']:g} {v['units']}{'' if v['verified'] else ' (unverified)'}" for k, v in record["constants"].items()}
    print(f"{machine}: {constants}; point table {'built' if record['point_table'].get('built') else 'from the cache'}")
display(pd.DataFrame(BUILD["counts"]).T)
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(PROFILE["checks"]).set_index("id"))

## 2. Sample points over the coils and the grid

The layout of the configuration in each machine's metres, over the coil elements (grey) and the
flux grid (dashed). Colour and label give the distance to the nearest coil element (check X4,
rule 0.15 m); the red cross is the reference point (0, 0).

In [ ]:
CLEARANCE = pd.read_csv(REPORT_DIR / "layout_clearance.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 7.5))
for ax, machine in zip(axes, ("DIII-D", "MAST")):
    cfg = machine_config(machine)
    geometry, source = cp.geometry_from_split(DATA_ROOT, machine)
    grid = GreenTable.load(GREEN_DIR / f"green_{cfg.file_prefix.split('_')[0]}.npz")
    for i in range(geometry.n_elements):
        o = element_outline(geometry.r[i], geometry.z[i], geometry.width[i], geometry.height[i],
                            geometry.angle1[i], geometry.angle2[i])
        ax.fill(o[:, 0], o[:, 1], color="0.55", lw=0)
    ax.add_patch(plt.Rectangle((grid.grid_r.min(), grid.grid_z.min()), np.ptp(grid.grid_r), np.ptp(grid.grid_z),
                               fill=False, ls="--", color="0.3"))
    rows = CLEARANCE[CLEARANCE["machine"] == machine]
    sc = ax.scatter(rows["R_m"], rows["Z_m"], c=rows["clearance_m"], cmap="viridis", s=45, edgecolor="k", zorder=3)
    for r in rows.itertuples(index=False):
        ax.annotate(f"{r.clearance_m:.2f}", (r.R_m, r.Z_m), fontsize=6, xytext=(3, 3), textcoords="offset points")
    pr, pz = CONFIG.layout.physical(cfg.major_radius.value, cfg.minor_radius.value)
    ax.plot(pr[-1], pz[-1], "r+", ms=12, mew=2)
    ax.set_aspect("equal")
    ax.set_xlabel("R (m)")
    ax.set_ylabel("Z (m)")
    ax.set_title(f"{machine}: R0 {cfg.major_radius.value} m, a0 {cfg.minor_radius.value} m", fontsize=10)
    fig.colorbar(sc, ax=ax, fraction=0.035, label="clearance (m)")
plt.show()
display(CLEARANCE.sort_values("clearance_m").groupby("machine").head(4)[
    ["machine", "column", "x", "y", "R_m", "Z_m", "clearance_m", "nearest_element", "ok"]])

## 3. Coil pattern

**Example frames, original and mirrored.** The coil points of a rising and a flat-top DIII-D
training frame and of a MAST frame, at their normalized positions, beside the mirrored frame
(`features.mirror_features`: points exchanged with their partners at (x, -y)). Check X5 compares
the mirrored points with points calculated from the A and B currents exchanged.

In [ ]:
def example_rows():
    rows = []
    for group, phase in (("train", "rising"), ("train", "flat_top"), ("mast_public_test", None)):
        t = PLOT[group].dropna(subset=POINTS)
        t = t if phase is None or not (t["phase"] == phase).any() else t[t["phase"] == phase]
        if len(t):
            rows.append((f"{LABEL[group]} {t.iloc[0]['shot']} frame {int(t.iloc[0]['frame'])} ({t.iloc[0]['phase']})", t.iloc[[0]]))
    return rows


XY = CONFIG.layout.points
EXAMPLES = example_rows()
fig, axes = plt.subplots(len(EXAMPLES), 2, figsize=(8, 3.4 * len(EXAMPLES)), squeeze=False)
for (title, row), (ax0, ax1) in zip(EXAMPLES, axes):
    original = row[POINTS].to_numpy(dtype=float)[0]
    mirrored = fe.mirror_features(row, CONFIG)[POINTS].to_numpy(dtype=float)[0]
    vmax = float(np.nanmax(np.abs(np.concatenate([original, mirrored])))) or 1.0
    for ax, values, name in ((ax0, original, "original"), (ax1, mirrored, "mirrored")):
        sc = ax.scatter(XY[:, 0], XY[:, 1], c=values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, s=140, edgecolor="k")
        ax.plot(0, 0, "k+", ms=10)
        ax.set_aspect("equal")
        ax.set_title(f"{name}: {title}", fontsize=8)
    fig.colorbar(sc, ax=[ax0, ax1], fraction=0.03, label="coil_pt")
plt.show()
print(f"X5: {[c['summary'] for c in PROFILE['checks'] if c['id'] == 'X5'][0]}")

**Named terms by current direction (X6).** The nine terms of the order-3 fit on positive-Ip and
negative-Ip DIII-D training frames. The coil pattern is divided by the signed Ip, so the two
directions are expected to overlap.

In [ ]:
train = PLOT["train"]
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
by_direction = {"positive Ip": train[train["ip_direction"] == 1], "negative Ip": train[train["ip_direction"] == -1]}
for ax, term in zip(axes.ravel(), TERMS):
    overlaid(ax, term, groups=tuple(by_direction), frames=by_direction)
axes[0, 0].legend(fontsize=7)
fig.suptitle("Named coil terms, DIII-D training frames by Ip direction")
plt.tight_layout()
plt.show()
display(pd.DataFrame(PROFILE["x6"]["table"]).set_index("feature").round(4))

**Correlations (X12).** Raw currents (as stored, `__mean`) beside the coil features, on the
same training frames; `abs_ip` is the magnitude of Ip. F42 found the raw currents strongly
correlated with each other and with Ip; dividing by Ip is meant to remove that shared factor.

In [ ]:
CORR = pd.read_parquet(REPORT_DIR / "correlations.parquet")
fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
for ax, kind in zip(axes, ("raw_currents", "coil_features")):
    c = CORR[CORR["kind"] == kind]
    names = list(dict.fromkeys(c["a"]))
    matrix = c.pivot(index="a", columns="b", values="r").loc[names, names]
    im = ax.imshow(matrix.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
    short = [n.replace("magnetics_", "").replace("__mean", "") for n in names]
    ax.set_xticks(range(len(names)), short, rotation=90, fontsize=6)
    ax.set_yticks(range(len(names)), short, fontsize=6)
    ax.set_title(f"{kind} ({int(c['n_rows'].iloc[0])} frames)", fontsize=10)
fig.colorbar(im, ax=axes.tolist(), fraction=0.02, label="r")
plt.show()
for kind in ("raw_currents", "coil_features"):
    s = PROFILE["x12"][kind]
    print(f"{kind}: median |r| {s['median_abs_r']:.3f}, largest {s['max_abs_r']:.3f} {s['max_pair']}; with Ip: "
          + ", ".join(f"{k} largest {v['max_abs_r']:.3f} ({v['max_column']})" for k, v in s["with_ip"].items()))

**The pattern after scaling (X13).** The coil points share one median and one interquartile
range (scale group `coil_pattern`), so scaling is the same affine map at every point and the
shape of each frame's pattern is kept: the scaled pattern correlates exactly with the unscaled
one. Scaling each point on its own would have changed the shape.

In [ ]:
rows = PLOT["train"].dropna(subset=POINTS)
rows = rows.iloc[fc.sample_rows(len(rows), 4, SEED)]
unscaled = rows[POINTS].to_numpy(dtype=float)
scaled = SCALER.transform(rows, "DIII-D", POINTS)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 3.8))
for k in range(len(rows)):
    label = f"{rows['shot'].iloc[k]} frame {int(rows['frame'].iloc[k])}"
    ax0.plot(unscaled[k], marker="o", label=label)
    ax1.plot(scaled[k], marker="o", label=label)
ax0.set_title("coil_pt, unscaled")
ax1.set_title("coil_pt, scaled with the shared statistics")
for ax in (ax0, ax1):
    ax.set_xticks(range(len(POINTS)), [f"({x:g},{y:g})" for x, y in XY], rotation=90, fontsize=6)
ax1.legend(fontsize=7)
plt.show()
center, scale = SCALER.source[POINTS[0]]
print(f"shared statistics: median {center:.5f}, interquartile range {scale:.5f}")
print("correlation of scaled with unscaled pattern per frame:",
      [round(float(np.corrcoef(u, s)[0, 1]), 6) for u, s in zip(unscaled, scaled) if np.ptp(u) > 0])

## 4. Thomson and field features

Electron pressure against the poloidal field pressure (`ts_beta_p`), the pressure peaking, the
cylindrical safety factor and, as a candidate, the Greenwald fraction, on DIII-D training,
DIII-D test and MAST. `q_cyl` on MAST uses the unverified MAST turn count (24); an error in it
shifts the whole MAST distribution by the same factor.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, column in zip(axes, ("ts_beta_p", "ts_pe_peaking", "q_cyl", "greenwald_frac")):
    overlaid(ax, column)
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()
summary = pd.DataFrame({LABEL[g]: {"ts_valid share": DATA[g]["ts_valid"].mean(),
                                   **{f"median {c}": DATA[g][c].median() for c in ("ts_beta_p", "ts_pe_peaking", "q_cyl", "greenwald_frac")}}
                        for g in COMPARED})
display(summary.round(4))

## 5. Time features (X15)

The 50 ms Ip slope in kA/s against the same slope relative to Ip in units of the resistive time
(`ip_rate_rel`), and the time since the Ip rise against `resistive_age`, on both machines. F33
found MAST plasmas changing several times faster in raw units; the scaled forms are meant to make
the two machines comparable. Then the share of frames with changing current on MAST (open item 6)
and the distribution of `ip_rise_ms` (10 percent crossing) per machine.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
overlaid(axes[0], "ip__rate50")
overlaid(axes[1], "ip_rate_rel")
axes[0].legend(fontsize=7)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for g in COMPARED:
    t = DATA[g].iloc[fc.sample_rows(len(DATA[g]), SCATTER_FRAMES, SEED)]
    since = (t["time_ms"] - t["ip_rise_ms"]) / 1000.0
    axes[0].scatter(since, t["resistive_age"], s=2, alpha=0.3, color=COLOR[g], label=LABEL[g])
    first = DATA[g].drop_duplicates("shot")["ip_rise_ms"].to_numpy(dtype=float)
    first = first[np.isfinite(first)]
    if first.size:
        axes[1].hist(first, bins=60, density=True, histtype="step", color=COLOR[g], label=f"{LABEL[g]} ({first.size} shots)")
axes[0].set_xlabel("time since the Ip rise (s)")
axes[0].set_ylabel("resistive_age")
axes[0].set_xscale("symlog", linthresh=0.01)
axes[0].set_yscale("symlog", linthresh=0.001)
axes[1].set_xlabel("ip_rise_ms")
for ax in axes:
    ax.legend(fontsize=7, markerscale=4)
plt.show()
x15 = PROFILE["x15"]
display(pd.DataFrame(x15["bands"]).set_index(["quantity", "machine"]).round(4))
display(pd.DataFrame(x15["changing_current"]).T.round(4))
display(pd.DataFrame(x15["ip_rise_ms"]).T.round(2))

## 6. Ranges (X8, X9, X11)

For each feature: the share of DIII-D test and MAST frames outside the training 1st to 99th
percentile band and outside the training range, and the median shift in training interquartile
ranges (`range_comparison.parquet`, phase `all`). Then the features of set B drawn on the three
splits, the two stretch terms by phase (X9: how far the model must extrapolate in elongation),
and the share of scaled values beyond 10 in magnitude (X11).

In [ ]:
RANGES = pd.read_parquet(REPORT_DIR / "range_comparison.parquet")
table = RANGES[(RANGES["phase"] == "all") & (RANGES["split"] != "validation")].pivot(
    index="feature", columns="split", values=["share_outside_band", "share_outside_range", "median_shift_iqr"])
table = table.loc[CONFIG.all_columns()]
with pd.option_context("display.max_rows", 100):
    display(table.sort_values(("share_outside_band", "mast_public_test"), ascending=False).round(3))

In [ ]:
SET_B = CONFIG.set_columns("set_b")
fig, axes = plt.subplots(4, 4, figsize=(16, 13))
for ax, column in zip(axes.ravel(), SET_B):
    overlaid(ax, column)
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for i, term in enumerate(("coil_c20", "coil_c02")):
    for j, phase in enumerate(("rising", "flat_top", "late")):
        frames = {g: PLOT[g][PLOT[g]["phase"] == phase] for g in COMPARED}
        overlaid(axes[i, j], term, frames=frames)
        axes[i, j].set_title(f"{term}, {phase}", fontsize=9)
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()
x9 = RANGES[RANGES["feature"].isin(["coil_c20", "coil_c02"]) & (RANGES["split"] != "validation")]
display(x9.pivot_table(index=["feature", "phase"], columns="split", values=["share_outside_band", "median_shift_iqr"]).round(3))

In [ ]:
extremes = pd.DataFrame(PROFILE["x11"]).T
display(extremes[(extremes > 0).any(axis=1)].round(5) if (extremes > 0).any().any() else "no scaled value beyond 10 on any split")

## 7. Combined distance (X10)

Distance in scaled feature space from each sampled frame to its nearest sampled DIII-D training
frame, per feature set, as a cumulative distribution per split. Dashed: with the mirrored
training frames added to the reference, as a model trained with mirroring sees them. The table
gives the percentiles and the share of each split beyond the validation 99th percentile; the
last table the median share of the squared distance carried by each feature group.

In [ ]:
NN = pd.read_parquet(REPORT_DIR / "nn_distance.parquet")
sets = list(dict.fromkeys(NN["set"]))
fig, axes = plt.subplots(1, len(sets), figsize=(6.5 * len(sets), 4.2), squeeze=False)
for ax, set_name in zip(axes[0], sets):
    for split, g in NN[NN["set"] == set_name].groupby("split", sort=False):
        for column, style in (("distance", "-"), ("distance_with_mirrors", "--")):
            d = np.sort(g[column].to_numpy())
            ax.plot(d, np.arange(1, d.size + 1) / d.size, style, color=COLOR[split],
                    label=LABEL[split] if style == "-" else None)
    ax.set_xscale("log")
    ax.set_xlabel("distance to the nearest training frame (scaled units)")
    ax.set_title(set_name)
    ax.legend(fontsize=7)
plt.show()
display(pd.DataFrame(PROFILE["x10"]["summary"]).set_index(["set", "split"]).round(3))
display(pd.DataFrame(PROFILE["x10"]["group_share_medians"]).set_index(["set", "split"]).round(3))

## 8. Thomson against force balance (X14)

`ts_beta_p` (Thomson route) against `shafranov_lambda` (the coils' vertical field, an estimate of
beta_p + li / 2 in the large-aspect-ratio form) with the line fitted per split. Agreement on
DIII-D with a shift on MAST alone points to an instrument or aspect-ratio effect in one of the
two routes; it is recorded as a reading, not acted on here.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=False, sharey=False)
fits = PROFILE["x14"]["all"]
for ax, g in zip(axes, COMPARED):
    t = DATA[g].iloc[fc.sample_rows(len(DATA[g]), SCATTER_FRAMES, SEED)]
    ax.scatter(t["ts_beta_p"], t["shafranov_lambda"], s=2, alpha=0.3, color=COLOR[g])
    x = t["ts_beta_p"].to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    if x.size:
        xs = np.linspace(*np.percentile(x, [1, 99]), 2)
        for ref, color, style in ((fits["train"], "k", "--"), (fits[g], COLOR[g], "-")):
            if ref["slope"] is not None and np.isfinite(ref["slope"]):
                ax.plot(xs, ref["intercept"] + ref["slope"] * xs, ls=style, color=color)
        lo, hi = np.percentile(x, [0.5, 99.5])
        if hi > lo:
            ax.set_xlim(lo, hi)
    y = t["shafranov_lambda"].to_numpy(dtype=float)
    y = y[np.isfinite(y)]
    if y.size and np.ptp(np.percentile(y, [0.5, 99.5])) > 0:
        ax.set_ylim(*np.percentile(y, [0.5, 99.5]))
    f = fits[g]
    ax.set_title(f"{LABEL[g]}: slope {f['slope'] if f['slope'] is not None else float('nan'):.3f}, "
                 f"offset {f['median_offset_from_train_line'] if f['median_offset_from_train_line'] is not None else float('nan'):+.3f}",
                 fontsize=9)
    ax.set_xlabel("ts_beta_p")
    ax.set_ylabel("shafranov_lambda")
plt.show()
display(pd.concat({k: pd.DataFrame(v).T for k, v in PROFILE["x14"].items()}).round(4))

## 9. Linear check (X16)

Ridges on the scaled features of each set, predicting the principal components of the plasma
target over mu0 R0 abs(Ip), scored with the organizers' scorer against `hybrid_ridge` on the same
frames (`scripts/feature_linear_check.py`). A score at or above the reference means the features
keep what a linear model uses in the raw inputs; a score well below it points to a group that
lost information, which the coefficient R2 with one group left out helps locate.

In [ ]:
LINEAR_PATH = REPORT_DIR / "linear_check.json"
if LINEAR_PATH.is_file():
    LINEAR = json.loads(LINEAR_PATH.read_text())
    scores = pd.DataFrame(LINEAR["scores"])
    columns = ["S", "r2_psi", "r2_qb", "one_minus_dlcfs", "consistency", "r2_R_axis", "r2_Z_axis", "r2_kappa",
               "r2_tri_top", "r2_tri_bot", "r2_volume", "r2_li"]
    display(scores.set_index(["subset", "model"])[columns].round(4))
    display(pd.DataFrame(LINEAR["status"]).T)
    display(pd.DataFrame(LINEAR["coefficient_r2"]).T.round(4))
else:
    LINEAR = None
    print(f"{LINEAR_PATH} not found; run python scripts/feature_linear_check.py")

## 10. Findings

The numbers behind the decisions this phase feeds (`docs/FEATURES.md`, "Decisions this phase
feeds"), gathered from the profile and the linear check.

In [ ]:
ranges_all = RANGES[RANGES["phase"] == "all"].set_index(["feature", "split"])
nn = pd.DataFrame(PROFILE["x10"]["summary"]).set_index(["set", "split"])
rows = []
for set_name in sets:
    rows.append({"decision": "coil form (points or named terms)", "evidence": f"X10 {set_name}: MAST median distance",
                 "value": nn.loc[(set_name, "mast_public_test"), "p50"]})
    rows.append({"decision": "coil form (points or named terms)", "evidence": f"X10 {set_name}: MAST share beyond validation 99th",
                 "value": nn.loc[(set_name, "mast_public_test"), "share_beyond_validation_p99"]})
if LINEAR is not None:
    for arm, s in LINEAR["status"].items():
        rows.append({"decision": "coil form (points or named terms)", "evidence": f"X16 {arm}: S minus hybrid_ridge", "value": s["delta_s"]})
for term in ("coil_c20", "coil_c02"):
    for phase in ("all", "flat_top"):
        r = RANGES[(RANGES["feature"] == term) & (RANGES["phase"] == phase) & (RANGES["split"] == "mast_public_test")]
        if len(r):
            rows.append({"decision": "extrapolation in elongation", "evidence": f"X9 {term} {phase}: MAST share outside training band",
                         "value": r["share_outside_band"].iloc[0]})
for feature in CONFIG.all_columns():
    if feature in POINTS:
        continue
    rows.append({"decision": "MAST's own statistics / candidates", "evidence": f"X8 {feature}: MAST share outside training band",
                 "value": ranges_all.loc[(feature, "mast_public_test"), "share_outside_band"]})
rows.append({"decision": "MAST's own statistics / candidates", "evidence": "X14 MAST median offset from the training line",
             "value": PROFILE["x14"]["all"]["mast_public_test"]["median_offset_from_train_line"]})
bands = pd.DataFrame(PROFILE["x15"]["bands"]).set_index(["quantity", "machine"])["width_over_reference"]
for quantity in ("ip__rate50 (kA/s)", "ip_rate_rel", "time since the Ip rise (s)", "resistive_age"):
    rows.append({"decision": "time features for transfer or DIII-D only", "evidence": f"X15 {quantity}: MAST band width over DIII-D",
                 "value": bands.get((quantity, "MAST"), np.nan)})
EVIDENCE = pd.DataFrame(rows)
with pd.option_context("display.max_rows", 100, "display.max_colwidth", None):
    display(EVIDENCE.round(4))

Readings: to be written after the review of sections 1 to 9, then carried to
`docs/findings.md` with the corrected mirroring entry (step 8).